# Reranking: Retrieve Broad, Score Narrow

| Field | Value |
|---|---|
| Stage | Retrieval and reranking |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
A reranker can only reorder candidates it receives. Candidate recall and reranker precision must be measured separately.

## 30-Second Summary

This notebook builds a transparent two-stage example where term frequency promotes a keyword-stuffed distractor. A bounded pairwise proxy penalizes repetition and restores the relevant MFA recovery document—only when candidate depth is at least two.

## Why This Matters

Fast first-stage retrieval is optimized for recall and can return shallow matches. More expensive query-document scoring can improve precision, but it cannot recover a missing document and adds latency.

## Scope

| Covers | Does not cover |
|---|---|
| Candidate depth, transparent reranking features, recall@k, MRR, latency boundary | Downloaded cross-encoder, LLM judge, production benchmark |


## Mental Model

```text
query -> fast retriever -> top-N candidates -> pairwise reranker -> top-k context
               recall gate ^                    precision gate
```


In [1]:
from collections import Counter
import re

query = "reset MFA after losing phone"
candidates = [
    {"id": "stuffed", "relevant": False, "text": "Reset reset MFA MFA lost phone keywords for a marketing taxonomy."},
    {"id": "recovery", "relevant": True, "text": "If your phone is lost, contact support to reset multi-factor authentication (MFA) after identity verification."},
    {"id": "password", "relevant": False, "text": "Reset a forgotten password from the account sign-in page."},
    {"id": "devices", "relevant": False, "text": "Register a new phone after signing in on an existing trusted device."},
]

def tokens(text: str) -> list[str]: return re.findall(r"[a-z0-9]+", text.lower())


## How It Works

Stage one cheaply scores query-term frequency. Stage two sees each query-document pair, rewards unique query coverage and an ordered lost-phone-to-reset relation, and penalizes repeated query terms. The proxy exposes mechanics; a trained cross-encoder learns richer interactions.


## Baseline

The first-stage scorer sums occurrences of query terms. Keyword stuffing therefore outranks the genuinely useful recovery instruction.


In [2]:
query_terms = tokens(query)
def first_stage_score(document: dict) -> int:
    counts = Counter(tokens(document["text"]))
    return sum(counts[term] for term in query_terms)

first_stage = sorted(candidates, key=lambda doc: (-first_stage_score(doc), doc["id"]))
[(doc["id"], first_stage_score(doc)) for doc in first_stage]


[('stuffed', 5), ('recovery', 4), ('devices', 2), ('password', 1)]

## Technique Implementation

The reranker uses only inspectable features: unique coverage, a recovery-instruction signal, and repetition penalty. It is deliberately not presented as a substitute for a trained relevance model.


In [3]:
def rerank_score(document: dict) -> float:
    document_tokens = tokens(document["text"])
    counts = Counter(document_tokens)
    unique_coverage = len(set(query_terms) & set(document_tokens)) / len(set(query_terms))
    recovery_signal = float("phone" in document_tokens and "reset" in document_tokens and "verification" in document_tokens)
    repetition_penalty = sum(max(0, counts[term] - 1) for term in set(query_terms)) / len(set(query_terms))
    return unique_coverage + 0.75 * recovery_signal - 0.5 * repetition_penalty

def rerank(candidate_set: list[dict]) -> list[dict]:
    return sorted(candidate_set, key=lambda doc: (-rerank_score(doc), doc["id"]))

[(doc["id"], round(rerank_score(doc), 3)) for doc in rerank(first_stage)]


[('recovery', 1.55), ('devices', 0.4), ('stuffed', 0.4), ('password', 0.2)]

## Controlled Experiment

We vary first-stage candidate depth from one to four. Recall@N asks whether the relevant document reaches stage two; reciprocal rank measures its final position after reranking.


In [4]:
experiment_rows = []
for candidate_depth in range(1, len(candidates) + 1):
    selected = first_stage[:candidate_depth]
    reranked = rerank(selected)
    first_relevant_rank = next((rank for rank, doc in enumerate(reranked, 1) if doc["relevant"]), None)
    experiment_rows.append({
        "candidate_depth": candidate_depth,
        "candidate_recall": float(any(doc["relevant"] for doc in selected)),
        "top_document": reranked[0]["id"],
        "reciprocal_rank": 1 / first_relevant_rank if first_relevant_rank else 0.0,
    })
experiment_rows


[{'candidate_depth': 1,
  'candidate_recall': 0.0,
  'top_document': 'stuffed',
  'reciprocal_rank': 0.0},
 {'candidate_depth': 2,
  'candidate_recall': 1.0,
  'top_document': 'recovery',
  'reciprocal_rank': 1.0},
 {'candidate_depth': 3,
  'candidate_recall': 1.0,
  'top_document': 'recovery',
  'reciprocal_rank': 1.0},
 {'candidate_depth': 4,
  'candidate_recall': 1.0,
  'top_document': 'recovery',
  'reciprocal_rank': 1.0}]

## Evaluation

At depth **1**, candidate recall is zero and reranking cannot help. At depth **2 or greater**, the recovery document is present and moves to rank 1 (MRR 1.0). The example proves the two-stage dependency, not real cross-encoder accuracy.


In [5]:
assert first_stage[0]["id"] == "stuffed"
assert experiment_rows[0]["candidate_recall"] == 0.0
assert all(row["top_document"] == "recovery" and row["reciprocal_rank"] == 1.0 for row in experiment_rows[1:])
assert rerank_score(next(doc for doc in candidates if doc["id"] == "recovery")) > rerank_score(first_stage[0])
print("Reranking checks passed.")


Reranking checks passed.


## Decision Guide

| Situation | Choice |
|---|---|
| Candidate recall already low | Improve first-stage retrieval first |
| Good recall, weak top-1 precision | Add evaluated reranker |
| Tight latency budget | Smaller candidate depth/model or no reranker |
| High-stakes ranking | Calibrated model plus human-reviewed hard negatives |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Reranker never finds answer | Relevant item absent | Raise/tune candidate recall |
| Latency spikes | Candidate depth/model too large | Batch, cap depth, timeout, fallback |
| Offline gains vanish | Easy or leaked labels | Add temporal/hard-negative evaluation |
| Pairwise scores misread as probabilities | Uncalibrated logits | Treat as ranks unless calibrated |


## Production Notes

### Observability
Track candidate recall, before/after rank, score deltas, depth, model/version, batch latency, and fallback rate.

### Safety and Guardrails
Never rerank unauthorized candidates; filtering must precede both stages.

### Latency and Cost
Measure p50/p95 by candidate depth and include timeout behavior in quality evaluation.


## Practice

Add a relevant document that ranks fifth in stage one. Plot final MRR against candidate depth and choose the smallest acceptable depth.

## Recall

Toggle - Recall: What is the reranker's hard limit?
It cannot select a document absent from the candidate set.

Toggle - Recall: Why evaluate candidate recall separately?
It distinguishes first-stage misses from ordering errors.

## Sources

- [BERT passage reranking paper](https://arxiv.org/abs/1901.04085)
- [Sentence Transformers cross-encoder reranking](https://www.sbert.net/examples/cross_encoder/applications/README.html)

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the two-stage dependency demonstration | Benchmark a trained reranker on shared hard negatives |
